## Imports

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.append("../../existing_metric_comparison/")
sys.path.append("../../../")
sys.path.append("../../utils/")
import data_cleanup
import metrics_lib as metrics
import os
import mapqc

In [ ]:
sc.set_figure_params(figsize=(5, 5))

## Set global variables

In [ ]:
study = "Sheppard_2020"
batch_variable = "sample"
kl = "1.0"
use_old_counts = True
if use_old_counts == True:
    suffix = "original_counts_"
else:
    suffix = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_{suffix}batch_{batch_variable}_kl{kl}_latent.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_{study}_counts.h5ad"
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/benchmark"
path_mapqc_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/atlas_extension/results/HLCA/{study}/{suffix}batch_{batch_variable}_kl{kl}/mapqc"

In [ ]:
if not os.path.exists(path_results_dir):
    if os.path.exists(os.path.dirname(path_results_dir)):
        print(f"Creating directory {path_results_dir}")
        os.mkdir(RESULT_DIR)
    else:
        print("Parent directory of path_results_dir does not exist. Create it first.")

In [ ]:
path_emb = "/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_Sheppard_2020_original_counts_batch_sample_kl1.0_latent.h5ad"

In [ ]:
path_counts = "/lustre/groups/ml01/workspace/lisa.sikkema/atlas_extension/data/HLCA/HLCA_and_Sheppard_2020_counts.h5ad"

## load and prep data

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)
adata_full = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

import mapqc results if wanted (for comparison):

In [ ]:
results_healthy = mapqc.load(os.path.join(path_mapqc_results_dir, "control.pkl"))
results_disease = mapqc.load(os.path.join(path_mapqc_results_dir, "case.pkl"))

## run metrics

In [ ]:
query_condition_1 = "Healthy"

In [ ]:
adata_1 = adata_full[
    (adata_full.obs.lung_condition == query_condition_1)
    | (adata_full.obs["r_or_q"] == "r")
].copy()

In [ ]:
del adata_full

subset query for metric functions below

In [ ]:
adata_query_1 = adata_1[adata_1.obs.study == "Sheppard_2020", :].copy()

set some variables for running metrics

In [ ]:
obsm_emb_key = "X_scanvi_emb"
donor_key = "donor_id"
dataset_key = "r_or_q"
sample_key = "sample"
original_emb_key = None
batch_key = "r_or_q"

get results for other metrics:

In [ ]:
scib_results_1 = metrics.calculate_scib_metrics(
    adata_1, obsm_emb_key, batch_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
symphony_results_1 = metrics.calculate_symphony_metrics(
    adata_1,
    adata_query_1,
    obsm_emb_key,
    donor_key,
    dataset_key,
    sample_key,
    dry_run=False,
)

In [ ]:
other_results_1 = metrics.calculate_other_metrics(
    adata_1, adata_query_1, obsm_emb_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
# save results
adata_1.obs["lisi_batch"] = scib_results["lisi_batch"]["per_cell"]
adata_1.obs["kbet_batch"] = scib_results["kbet_batch"]["per_cell"]
adata_1.obs["lisi_query_donor"] = symphony_results["lisi_query_donor"]["per_cell"]
adata_1.obs["wiq_knn_corr"] = symphony_results["wiq_knn_corr"]["per_cell"]
adata_1.obs["cluster_preservation_score"] = other_results["cluster_preservation_score"][
    "per_cell"
]

In [ ]:
adata_1.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
].to_csv(os.path.join(path_results_dir, f"alt_metrics_{query_condition_1.lower()}.csv"))

## Visualize 

In [ ]:
sc.pl.umap(
    adata_1, color="r_or_q", frameon=False, palette={"r": "grey", "q": "dodgerblue"}
)

Run Load and prep embeddings and set global variables first

In [ ]:
adata_1.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
] = pd.read_csv(
    os.path.join(path_results_dir, f"alt_metrics_{query_condition_1.lower()}.csv"),
    index_col="Unnamed: 0",
).loc[
    adata_1.obs.index, :
]

In [ ]:
for metric in [
    "lisi_batch",
    "kbet_batch",
    "lisi_query_donor",
    "wiq_knn_corr",
    "cluster_preservation_score",
]:
    adata_1.obs.loc[adata_1.obs.core_or_extension == "core", metric] = np.nan
    sc.pl.umap(adata_1, color=metric)  # ,vmax='p99')

Quantification/visualization different metrics:

In [ ]:
hist_kwargs = {
    "wiq_knn_corr": {"range": (-1, 1), "bins": 10},
    "cluster_preservation_score": {"bins": 10},
    "lisi_query_donor": {"bins": 10},
}
xlabels = {
    "wiq_knn_corr": "Within-query k-NN-correlation\n(high is good)",
    "cluster_preservation_score": "Cluster preservation score\n(high is good)",
    "lisi_query_donor": "Query donor LISI\n(high is good)",
}
results = {}

In [ ]:
fignumber = 0
n_figures = 4
n_cols = 4
n_rows = int(np.ceil(n_figures / n_cols))
fig = plt.figure(figsize=(4 * n_cols, 3 * n_rows))
fig_count = 1

# start with mapqc scores
ax = fig.add_subplot(n_rows, n_cols, fig_count)
mapqc_scores = results_healthy.adata.obs.mapqc_score[
    ~pd.isnull(results_healthy.adata.obs.mapqc_score)
]
xmax = 32
# check that we are not ignoring data
if np.percentile(mapqc_scores, 99) > xmax:
    raise ValueError("You are ignoring more than 1% of your data. Change your xmax.")
ax.hist(
    mapqc_scores, bins=range(0, xmax, 2), range=(0, xmax), rwidth=0.9
)  # density=True,
ax.set_xlabel("MapQC score\n(low is good)")
ax.set_ylabel("N cells")  # Fraction of cells")
ax.spines[["top", "right"]].set_visible(False)
ax.vlines(x=2, ymin=0, ymax=13000, color="red", linestyle="--")
plt.grid(False)
results["mapqc"] = {"mean": np.mean(mapqc_scores), "median": np.median(mapqc_scores)}
fig_count += 1

# then loop through other metrics
for metric in ["wiq_knn_corr", "cluster_preservation_score", "lisi_query_donor"]:
    ax = fig.add_subplot(n_rows, n_cols, fig_count)
    metric_vals = adata_1.obs[metric][~pd.isnull(adata_1.obs[metric])]
    ax.hist(metric_vals, rwidth=0.9, **hist_kwargs[metric])
    ax.set_ylabel("N cells")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(xlabels[metric])
    fig_count += 1
    plt.grid(False)
    results[metric] = {"mean": np.mean(metric_vals), "median": np.median(metric_vals)}
plt.tight_layout()
plt.show()
plt.close()
for metric, res in results.items():
    print(f"{metric}: {res}")

### Query disease

In [ ]:
query_condition_2 = "IPF"

In [ ]:
adata_2 = adata_full[
    (adata_full.obs.lung_condition == query_condition_2)
    | (adata_full.obs["r_or_q"] == "r")
].copy()

In [ ]:
del adata_full

subset query for metric functions below

In [ ]:
adata_query_2 = adata_2[adata_2.obs.study == "Sheppard_2020", :].copy()

set some variables for running metrics

In [ ]:
obsm_emb_key = "X_scanvi_emb"
donor_key = "donor_id"
dataset_key = "r_or_q"
sample_key = "sample"
original_emb_key = None
batch_key = "r_or_q"

get results for other metrics:

In [ ]:
scib_results_2 = metrics.calculate_scib_metrics(
    adata_2, obsm_emb_key, batch_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
symphony_results_2 = metrics.calculate_symphony_metrics(
    adata_2,
    adata_query_2,
    obsm_emb_key,
    donor_key,
    dataset_key,
    sample_key,
    dry_run=False,
)

In [ ]:
other_results_2 = metrics.calculate_other_metrics(
    adata_2, adata_query_2, obsm_emb_key, dataset_key, sample_key, dry_run=False
)

In [ ]:
# save results
adata_2.obs["lisi_batch"] = scib_results_2["lisi_batch"]["per_cell"]
adata_2.obs["kbet_batch"] = scib_results_2["kbet_batch"]["per_cell"]
adata_2.obs["lisi_query_donor"] = symphony_results_2["lisi_query_donor"]["per_cell"]
adata_2.obs["wiq_knn_corr"] = symphony_results_2["wiq_knn_corr"]["per_cell"]
adata_2.obs["cluster_preservation_score"] = other_results_2[
    "cluster_preservation_score"
]["per_cell"]

In [ ]:
adata_2.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
].to_csv(os.path.join(path_results_dir, f"alt_metrics_{query_condition_2.lower()}.csv"))

## Visualize 

Run Load and prep embeddings and set global variables first

In [ ]:
adata_2.obs[
    [
        "lisi_batch",
        "kbet_batch",
        "lisi_query_donor",
        "wiq_knn_corr",
        "cluster_preservation_score",
    ]
] = pd.read_csv(
    os.path.join(path_results_dir, f"alt_metrics_{query_condition_2.lower()}.csv"),
    index_col="Unnamed: 0",
)

In [ ]:
for metric in [
    "lisi_batch",
    "kbet_batch",
    "lisi_query_donor",
    "wiq_knn_corr",
    "cluster_preservation_score",
]:
    adata_2.obs.loc[adata_2.obs.core_or_extension == "core", metric] = np.nan
    sc.pl.umap(adata_2, color=metric)  # ,vmax='p99')